In [1]:
import pandas as pd
from math import cos, radians, sqrt
from sklearn.neighbors import BallTree
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import plotly.express as px
import warnings
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.decomposition import PCA
from scipy.stats import skew
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import pearsonr

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
warnings.filterwarnings('ignore')

In [3]:
raw = "C:\\Users\\Taavi\\Desktop\\BPhil\\Raw data\\"
clean = "C:\\Users\\Taavi\\Desktop\\BPhil\\Clean data\\"

### Variables to use for comparison (comparing R-squared in regression):
- Condemnations
- Vacancies
- QoL
- Age (from Weaver & Bagchi-Sen - although they did this as % pop 25+ no HS)
- Unemployment (from Weaver & Bagchi-Sen)
- Poverty (from Weaver & Bagchi-Sen)
- Income
- Race

# Import & prepare data

#### parcels, viols, and crimes (I'll have to sort out the crimes 'tract' another time)

In [4]:
parcels = pd.read_csv(clean + 'blight.csv')
parcels['tract'] =  parcels['tract'].astype(str).str.split('.').str[0].astype(int)

viols = pd.read_csv(clean + 'viols_ready_to_measure.csv')
#crimes = pd.read_csv(clean + 'clean_crimes.csv')

In [5]:
viols = (
    viols
    .merge(parcels[['parcelID', 'tract']], on = 'parcelID', how = 'inner')
)

In [6]:
# crimes['tract'] = np.where(crimes['tract'].astype(str).str.split('.').str[0].str.len() < 6, '00000' + crimes['tract'].astype(str), crimes['tract'])
# crimes['tract'] = crimes['tract'].str[-6:-2]
# crimes['tract'] = crimes['tract'] + '00'

In [7]:
n_parcels_by_tract = parcels.groupby('tract')['parcelID'].nunique().reset_index().rename(columns = {'parcelID': 'num_parcels'})
n_parcels_by_tract = n_parcels_by_tract.query('num_parcels >= 20')
viols = viols.merge(n_parcels_by_tract, on = 'tract', how = 'left')
#crimes = crimes.merge(n_parcels_by_tract, on = 'tract', how = 'left')

#### condemnations, age, employment, poverty, income, race, sex

In [8]:
conds = pd.read_csv(clean + 'clean_condemnations.csv')
age = pd.read_csv(clean + 'age_ACS_data.csv')
empl = pd.read_csv(clean + 'employment_ACS_data.csv')
pov = pd.read_csv(clean + 'poverty_ACS_data.csv')
inc = pd.read_csv(clean + 'income_ACS_data.csv')
race = pd.read_csv(clean + 'race_ACS_data.csv')
sex = pd.read_csv(clean + 'sex_ACS_data.csv')

In [9]:
conds = conds.merge(parcels[['parcelID', 'tract']], on = 'parcelID', how = 'left').merge(n_parcels_by_tract, on = 'tract', how = 'left')
conds = conds.drop(columns = 'year')
conds = conds.dropna()

age = age.merge(n_parcels_by_tract, on = 'tract', how = 'left').dropna()
empl = empl.merge(n_parcels_by_tract, on = 'tract', how = 'left').dropna()
pov = pov.merge(n_parcels_by_tract, on = 'tract', how = 'left').dropna()
inc = inc.merge(n_parcels_by_tract, on = 'tract', how = 'left').dropna()
race = race.merge(n_parcels_by_tract, on = 'tract', how = 'left').dropna()
sex = sex.merge(n_parcels_by_tract, on = 'tract', how = 'left').dropna()

# Measuring census tract X-per-unit

#### BPU (code violations)

In [10]:
viols = viols.query('num_parcels >= 20')

In [11]:
m1_viols = (
    viols.groupby('tract')['violID'].count() /
    viols.groupby('tract')['num_parcels'].first()
).reset_index().rename(columns = {0: 'm1_viols'})

#### Sqrt-transforming BPU

In [12]:
m1_viols = m1_viols.assign(m1_viols = np.sqrt(m1_viols['m1_viols']))

#### condemnations

In [13]:
m1_conds = (
    conds.groupby('tract')['parcelID'].count() /
    conds.groupby('tract')['num_parcels'].first()
).reset_index().rename(columns = {0: 'm1_conds'})

model = smf.ols(formula = 'm1_conds ~ m1_viols', data = m1_viols.merge(m1_conds, on = 'tract', how = 'left')).fit()
print(f'M1: conds ~ viols - R-squared: {round(model.rsquared, 2)*100}%')

M1: conds ~ viols - R-squared: 28.000000000000004%


In [14]:
m1_viols.merge(m1_conds, on = 'tract', how = 'left').iloc[:, 1:].corr().iloc[1:, :1]

,m1_viols
m1_conds,0.53


In [15]:
x = m1_viols.merge(m1_conds, on = 'tract', how = 'left')
x['m1_viols_sqrt'] = np.sqrt(x['m1_viols'])
x['m1_viols_log'] = np.log1p(x['m1_viols'])
x['m1_conds_sqrt'] = np.sqrt(x['m1_conds'])
x['m1_conds_log'] = np.log1p(x['m1_conds'])

In [16]:
pearsonr(x.dropna()['m1_viols_sqrt'], x.dropna()['m1_conds_sqrt'])

PearsonRResult(statistic=np.float64(0.5618410843848186), pvalue=np.float64(3.6065056218033216e-11))

In [17]:
x[['m1_viols_sqrt', 'm1_conds_sqrt']].corr()

,m1_viols_sqrt,m1_conds_sqrt
m1_viols_sqrt,1.00,0.56
m1_conds_sqrt,0.56,1.00


In [18]:
model = smf.ols(formula = 'm1_conds_sqrt ~ m1_viols_sqrt', data = x).fit()
print(f'M1: conds_sqrt ~ viols_sqrt - R-squared: {round(model.rsquared, 2)*100}%')

M1: conds_sqrt ~ viols_sqrt - R-squared: 32.0%


In [19]:
x[['m1_viols_log', 'm1_conds_log']].corr()

,m1_viols_log,m1_conds_log
m1_viols_log,1.00,0.55
m1_conds_log,0.55,1.00


In [20]:
model = smf.ols(formula = 'm1_conds_log ~ m1_viols_log', data = x).fit()
print(f'M1: conds_log ~ viols_log - R-squared: {round(model.rsquared, 2)*100}%')

M1: conds_log ~ viols_log - R-squared: 30.0%


In [24]:
x.to_csv(clean + 'BPU_blight.csv', index = False)

#### sex

In [173]:
m1_male = (
    sex.groupby('tract')['male'].median() /
    sex.groupby('tract')['num_parcels'].first()
).reset_index().rename(columns = {0: 'm1_male'})

model = smf.ols(formula = 'm1_male ~ m1_viols', data = m1_viols.merge(m1_male, on = 'tract', how = 'left')).fit()
print(f'M1: male ~ viols - R-squared: {round(model.rsquared, 2)*100}%')

m1_female = (
    sex.groupby('tract')['female'].median() /
    sex.groupby('tract')['num_parcels'].first()
).reset_index().rename(columns = {0: 'm1_female'})

model = smf.ols(formula = 'm1_female ~ m1_viols', data = m1_viols.merge(m1_female, on = 'tract', how = 'left')).fit()
print(f'M1: female ~ viols - R-squared: {round(model.rsquared, 2)*100}%')

M1: male ~ viols - R-squared: 3.0%
M1: female ~ viols - R-squared: 0.0%


#### race

In [174]:
race.head(1)

,total_pop,white,black,native,asian,islander,other,mixed,tract,block_group,num_parcels
0,1967,0.69,0.20,0.00,0.07,0.00,0.00,0.04,20100,1,1614.00


In [ ]:
m1_white = (
    race.groupby('tract')['white'].median() /
    race.groupby('tract')['num_parcels'].first()
).reset_index().rename(columns = {0: 'm1_white'})

model = smf.ols(formula = 'm1_white ~ m1_viols', data = m1_viols.merge(m1_white, on = 'tract', how = 'left')).fit()
print(f'M1: white ~ viols - R-squared: {round(model.rsquared, 2)*100}%')

m1_black = (
    race.groupby('tract')['black'].median() /
    race.groupby('tract')['num_parcels'].first()
).reset_index().rename(columns = {0: 'm1_black'})

model = smf.ols(formula = 'm1_black ~ m1_viols', data = m1_viols.merge(m1_black, on = 'tract', how = 'left')).fit()
print(f'M1: black ~ viols - R-squared: {round(model.rsquared, 2)*100}%')

m1_white = (
    race.groupby('tract')['white'].median() /
    race.groupby('tract')['num_parcels'].first()
).reset_index().rename(columns = {0: 'm1_white'})

model = smf.ols(formula = 'm1_white ~ m1_viols', data = m1_viols.merge(m1_white, on = 'tract', how = 'left')).fit()
print(f'M1: white ~ viols - R-squared: {round(model.rsquared, 2)*100}%')

m1_black = (
    race.groupby('tract')['black'].median() /
    race.groupby('tract')['num_parcels'].first()
).reset_index().rename(columns = {0: 'm1_black'})

model = smf.ols(formula = 'm1_black ~ m1_viols', data = m1_viols.merge(m1_black, on = 'tract', how = 'left')).fit()
print(f'M1: black ~ viols - R-squared: {round(model.rsquared, 2)*100}%')

M1: white ~ viols - R-squared: 2.0%
M1: black ~ viols - R-squared: 0.0%
